# Non-Local LLM Inference

## Objectives

- Set up API credentials safely.
- Force models to return **structured** output instead of free text.
- Handle **retries / rate limits** so a flaky network doesn't kill a long run.
- Track **token usage and cost** before scaling.

[delete after completing]

In [ ]:
import os

from pathlib import Path
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from google import genai
from google.genai import types
import anthropic

API_KEY:  
API_SECRET:  


----
# Non-Local Calls to LLMs
Non-local calls are one way to interact with large language models. You send a request over the internet to a provider's servers (e.g., OpenAI, Anthropic, Google) and receive a generated response back. 


In [ ]:
load_dotenv()

## Construct the Prompt


In [ ]:
# cases used in ppt examples (df row will be csv row - 2 -- index and column headings)
# 136 = prompt
# 207 = prompt
# 223 = not a prompt
# 3558 = not a prompt
CASE = "Okay, and what would you do?"

PROMPT = f"Classify the following utterance as either a 1 = dialogic prompt or 0 = not a dialogic prompt. A dialogic prompt is defined as an utterance that implies, encourages, requests, or expects a new speaker (or multiple new speakers) to make a verbal contribution. Here is the utterance: \"{CASE}\" Return only 0 or 1."


another way to do the above:

In [ ]:
# ---- get prompt codebook ----
path_to_prompts = AIMECON_DIR / "data_management" / "prompt_codebook.xlsx"
prompts = pd.read_excel(path_to_prompts)

# ---- get data ----
path_to_data = TRAIN_FILE
df = pd.read_excel(path_to_data)

CASE = df.loc[134, "text"]

PROMPT = (prompts.loc[prompts.id == "Coding", "prompt"].item() + 
          prompts.loc[prompts.id == "Construct", "prompt"].item() +
          prompts.loc[prompts.id == "Prompt1", "prompt"].item() +
          f"\"{CASE}\"" + 
          prompts.loc[prompts.id == "Format", "prompt"].item()
          )

# ⚙️ Model Settings

**System Prompt**

System prompts (also called System messages) are persistent instructions for how the model should behave. Once set, they guide every subsequent interaction.

how do these work in an api context? do they persist with the token?

**User Prompt**

User

**Model**


**Temperature**

A parameter that controls the randomness of text generated by LLMs during inference. Each token is assigned a probability of occurance based on the prompt and the tokens that come before it. Temperature modifies this probability distribution such that at higher temperatures increase the likelihood of selecting less probable tokens.


**Top P**

An alternative to sampling with temperature, called nucleus sampling, where the model considers the results of the tokens with `top_p` probability mass. So 0.1 means only the tokens comprising the top 10% probability mass are considered.

**Token Limits**

The maximum number of tokens output by the model





In [ ]:
GPT_MODEL = "gpt-5.6-terra" # https://developers.openai.com/api/docs/models/all
CONTEXT = "You are an educational researcher" # SYSTEM PROMPT
TOKENS = 100
TEMPERATURE = 0.1

# cost per million tokens
IN_RATE = 2.00
OUT_RATE = 12.00

In [ ]:
# ---- GPT ----
# get api key: https://platform.openai.com/api-keys

# check that the api key got loaded in the .env
# if loaded, prints the key
# if not loaded, prints ERROR
key = "OPENAI_API_KEY"
print(os.environ.get(key, f"ERROR: Variable {key} Not Found"))

# initialize model
openai = OpenAI()

# format prompt
prompt = [
    #{"role": "system", "content": CONTEXT},
    {"role": "user", "content": PROMPT}
  ]

# model settings
# https://developers.openai.com/api/reference/resources/chat/subresources/completions/methods/create
prompt_gpt = openai.chat.completions.create(
    model = GPT_MODEL, 
    messages = prompt,
    temperature = TEMPERATURE # range = 0-2
    )

response = prompt_gpt.choices[0].message.content
# https://developers.openai.com/api/docs/models/gpt-4o
in_rate = 2.50
out_rate = 10.00
input_cost = prompt_gpt.usage.prompt_tokens / 1_000_000 * in_rate
output_cost = prompt_gpt.usage.completion_tokens / 1_000_000 * out_rate

print(f"{GPT_MODEL} classified this as a {response}")
print(f"Prompt Tokens = {prompt_gpt.usage.prompt_tokens} (${input_cost})")
print(f"Completion Tokens = {prompt_gpt.usage.completion_tokens} (${output_cost})")

# 📋 Formatting Output
json


# ⏭️ Retries & Rate Limits

# 💰 Token Usage & Cost